# 🏷️ Sessão 08 — Classificação de Sítio via Deep Learning

> **Objetivo:** sair pela primeira vez da regressão e atacar um problema de **classificação multiclasse** — predizer a classe de qualidade do sítio (I, II, III) a partir de variáveis dendrométricas. Comparar o MLP contra dois baselines pragmáticos (regra majoritária e árvore de decisão) e estabelecer quando deep learning agrega valor real à classificação tabular.

---

## 📑 Sumário

1. [Da regressão para a classificação](#1-da-regressão-para-a-classificação)
2. [Setup e Preparação](#2-setup-e-preparação)
3. [Baselines Clássicos](#3-baselines-clássicos)
4. [MLP Classificador](#4-mlp-classificador)
5. [Matriz de Confusão](#5-matriz-de-confusão)
6. [Métricas por Classe](#6-métricas-por-classe)
7. [Análise de Confiança via Probabilidades](#7-análise-de-confiança-via-probabilidades)
8. [Síntese: ML em Classificação Tabular](#8-síntese-ml-em-classificação-tabular)

## 1. Da regressão para a classificação

### Mudanças conceituais

| Aspecto | Regressão (sessões 06–07) | Classificação (esta sessão) |
|---|---|---|
| Saída do modelo | 1 número contínuo (altura, volume) | k logits (um por classe) |
| Função de perda | MSE — distância ao alvo | Cross-Entropy — log-likelihood negativo |
| Predição final | valor da saída | argmax dos logits → índice da classe |
| Métrica principal | RMSE, R² | Acurácia, F1, matriz de confusão |

### Por que cross-entropy?

Para classificação multiclasse, a função de perda padrão é:

$$\mathcal{L} = -\sum_{i=1}^{N} \sum_{k=1}^{K} y_{i,k} \log\hat{p}_{i,k}$$

onde $y_{i,k} = 1$ se a instância $i$ pertence à classe $k$ e $\hat{p}_{i,k}$ é a probabilidade predita. A função penaliza fortemente predições confiantes mas erradas, e é matematicamente equivalente à **máxima verossimilhança** sob a distribuição categórica.

## 2. Setup e Preparação

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

from forestpy.data.loaders import load_pef_vinhedo
from forestpy.ml.preprocessing import StandardScalerForest
from forestpy.ml.mlp import MLPClassifier, MLPClassifierTrainer
from forestpy.ml.metrics import (
    classification_report,
    confusion_matrix,
    precision_per_class,
    recall_per_class,
    f1_per_class,
)
from forestpy.utils import set_seed, get_logger
from forestpy.viz.style import apply_forest_style
from forestpy.viz.diagnostics import plot_learning_curve
from forestpy.viz.classification import plot_confusion_matrix, plot_class_metrics

set_seed(42)
apply_forest_style()
log = get_logger('sessao_08')

df = load_pef_vinhedo(synthetic_fallback=True, n_synthetic=500)
log.info(f'Dataset: {df.shape[0]} árvores')

In [ ]:
# Codificação das classes: I=0, II=1, III=2
classe_map = {'I': 0, 'II': 1, 'III': 2}
classe_names = ['I', 'II', 'III']
y = np.array([classe_map[c] for c in df['classe']])

X = df[['dap', 'h', 'idade']].values.astype(np.float32)

log.info(f'Features: dap, h, idade — shape {X.shape}')
log.info(f'Distribuição das classes (n por classe):')
for nome, count in zip(classe_names, np.bincount(y)):
    log.info(f'  Classe {nome}: {count}')

In [ ]:
# Split treino/teste 80/20
rng = np.random.default_rng(42)
idx = rng.permutation(len(df))
tr_idx, te_idx = idx[:400], idx[400:]

X_tr_raw, X_te_raw = X[tr_idx], X[te_idx]
y_tr, y_te = y[tr_idx], y[te_idx]

# Normalização (fit no treino)
scaler = StandardScalerForest()
X_tr = scaler.fit_transform(X_tr_raw).astype(np.float32)
X_te = scaler.transform(X_te_raw).astype(np.float32)

log.info(f'Treino: {len(X_tr)} | Teste: {len(X_te)}')

## 3. Baselines Clássicos

Avaliamos dois pontos de referência antes do MLP:

1. **Regra majoritária**: sempre prediz a classe mais frequente. Estabelece o piso (chão estatístico).
2. **Árvore de Decisão**: modelo clássico interpretável, padrão em classificação tabular.

In [ ]:
# Baseline 1: regra majoritária
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_tr_raw, y_tr)
pred_dummy = dummy.predict(X_te_raw)
report_dummy = classification_report(y_te, pred_dummy)

# Baseline 2: árvore de decisão
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_tr_raw, y_tr)
pred_tree = tree.predict(X_te_raw)
report_tree = classification_report(y_te, pred_tree)

log.info('Regra majoritária:')
for k, v in report_dummy.items():
    log.info(f'  {k:20s} = {v:.4f}')
log.info('')
log.info('Árvore de Decisão (sklearn, profundidade 5):')
for k, v in report_tree.items():
    log.info(f'  {k:20s} = {v:.4f}')

## 4. MLP Classificador

Arquitetura: **3 features → 32 → 16 → 3 classes** com dropout 0.15.

In [ ]:
model = MLPClassifier(input_dim=3, hidden_dims=[32, 16], n_classes=3, dropout=0.15)
log.info(f'Parâmetros treináveis: {model.count_parameters()}')

# Validação interna para early stopping
n_val = int(0.2 * len(X_tr))
trainer = MLPClassifierTrainer(model, learning_rate=1e-3, weight_decay=1e-5)
history = trainer.fit(
    X_tr[:-n_val], y_tr[:-n_val],
    X_tr[-n_val:], y_tr[-n_val:],
    epochs=200, batch_size=32, patience=25, verbose=False,
)
log.info(f'Treino: {len(history.train_loss)} épocas (melhor: {history.best_epoch})')

In [ ]:
fig_lc = plot_learning_curve(
    history.train_loss, history.val_loss,
    best_epoch=history.best_epoch,
    title='Curva de Aprendizado — MLP Classificador',
)
fig_lc.savefig('../reports/figures/08_curva_aprendizado.png')
plt.show()

In [ ]:
# Avaliação no holdout
pred_mlp = trainer.predict(X_te)
report_mlp = classification_report(y_te, pred_mlp)

log.info('MLP Classificador (teste):')
for k, v in report_mlp.items():
    log.info(f'  {k:20s} = {v:.4f}')

In [ ]:
# Tabela comparativa
comparativo = pd.DataFrame([
    {'Modelo': 'Regra Majoritária', **report_dummy},
    {'Modelo': 'Árvore de Decisão', **report_tree},
    {'Modelo': 'MLP', **report_mlp},
]).round(4)

comparativo.to_csv('../reports/tables/08_classificacao_sitio.csv', index=False)
comparativo

## 5. Matriz de Confusão

A matriz revela **quais erros** o modelo comete — informação muito mais rica que a acurácia agregada.

In [ ]:
fig_cm, axes = plt.subplots(1, 2, figsize=(15, 6))

# Matriz absoluta (contagens)
for ax, (titulo, y_pred) in zip(axes, [
    ('Árvore de Decisão', pred_tree),
    ('MLP', pred_mlp),
], strict=False):
    cm = confusion_matrix(y_te, y_pred, labels=[0, 1, 2])
    im = ax.imshow(cm, cmap='Greens', vmin=0)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(classe_names)
    ax.set_yticklabels(classe_names)
    ax.set_xlabel('Predito')
    ax.set_ylabel('Observado')
    ax.set_title(titulo, fontweight='bold')
    threshold = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(3):
        for j in range(3):
            cor = 'white' if cm[i, j] > threshold else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color=cor, fontsize=12, fontweight='bold')

fig_cm.suptitle('Matrizes de Confusão — Comparação de Modelos',
                fontsize=14, fontweight='bold')
fig_cm.tight_layout()
fig_cm.savefig('../reports/figures/08_matrizes_confusao.png')
plt.show()

In [ ]:
# Matriz de confusão normalizada (% por linha = recall por classe)
fig_cm_norm = plot_confusion_matrix(
    y_te, pred_mlp, labels=[0, 1, 2],
    normalize=True,
    title='Matriz de Confusão Normalizada — MLP',
)
fig_cm_norm.savefig('../reports/figures/08_matriz_confusao_normalizada.png')
plt.show()

**🔍 Leitura da matriz normalizada:**
- Diagonal = taxa de acerto por classe (recall)
- Erros adjacentes (classe I prevista como II, ou III como II) indicam que o modelo está **na vizinhança correta** — natural para classes ordinais como qualidade de sítio
- Erros distantes (I prevista como III) seriam mais graves

## 6. Métricas por Classe

In [ ]:
# Métricas detalhadas por classe — MLP
p_mlp = precision_per_class(y_te, pred_mlp, labels=[0, 1, 2])
r_mlp = recall_per_class(y_te, pred_mlp, labels=[0, 1, 2])
f_mlp = f1_per_class(y_te, pred_mlp, labels=[0, 1, 2])

fig_metrics = plot_class_metrics(
    p_mlp, r_mlp, f_mlp, classe_names,
    title='Precision, Recall e F1 por Classe — MLP',
)
fig_metrics.savefig('../reports/figures/08_metricas_por_classe.png')
plt.show()

# Tabela detalhada
pd.DataFrame({
    'Classe': classe_names,
    'Precision': p_mlp,
    'Recall': r_mlp,
    'F1': f_mlp,
}).round(4)

## 7. Análise de Confiança via Probabilidades

Diferentemente de modelos discretos, o MLP fornece **probabilidades** para cada classe (após softmax). Isso permite analisar a **confiança** das predições — útil para identificar casos onde o modelo "hesita" e priorizar verificação manual em campo.

In [ ]:
probs = trainer.predict_proba(X_te)

# Probabilidade da classe predita (confiança)
confianca = probs.max(axis=1)

log.info(f'Confiança média do MLP: {confianca.mean():.3f}')
log.info(f'Confiança mínima: {confianca.min():.3f}')
log.info(f'Confiança máxima: {confianca.max():.3f}')

# Acurácia em função do threshold de confiança
thresholds = np.arange(0.4, 1.0, 0.05)
linhas = []
for t in thresholds:
    mask = confianca >= t
    if mask.sum() > 0:
        acc = (pred_mlp[mask] == y_te[mask]).mean()
        linhas.append({
            'Threshold': round(t, 2),
            'Cobertura (%)': round(100 * mask.mean(), 1),
            'Acurácia entre cobertos': round(acc, 3),
        })
pd.DataFrame(linhas)

In [ ]:
# Visualização: acurácia condicional vs. cobertura
df_cov = pd.DataFrame(linhas)
fig_cov, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_cov['Cobertura (%)'], df_cov['Acurácia entre cobertos'],
        marker='o', lw=2, color='#2d5016')
ax.set_xlabel('Cobertura (% das amostras incluídas)')
ax.set_ylabel('Acurácia entre amostras incluídas')
ax.set_title('Trade-off Confiança × Cobertura — MLP', fontweight='bold')
ax.invert_xaxis()  # mais à direita = mais inclusivo (threshold mais baixo)
ax.grid(True, alpha=0.3)
fig_cov.tight_layout()
fig_cov.savefig('../reports/figures/08_confianca_cobertura.png')
plt.show()

**🔍 Como ler:** se aceitarmos automatizar apenas as predições do MLP com confiança ≥ 80%, qual fração das amostras o modelo classifica sozinho e com que acurácia? Esse trade-off é central em produção real, onde **predições de baixa confiança** são encaminhadas para revisão humana.

## 8. Síntese: ML em Classificação Tabular

### Resultado central

Comparando os três modelos no holdout de teste, o **MLP empata com a árvore de decisão** e ambos batem confortavelmente a regra majoritária. Esse cenário é frequente em classificação tabular com poucas features — **deep learning não é dominante neste regime**.

### Por que esse resultado é didaticamente importante?

A literatura recente (Shwartz-Ziv & Armon, 2022) documenta sistematicamente que árvores e ensembles (Random Forest, Gradient Boosting) tendem a vencer redes neurais em dados tabulares de tamanho moderado. Isso ocorre porque:

1. Árvores capturam interações com poucas features de forma eficiente
2. Não exigem normalização ou engenharia de features
3. Possuem viés indutivo adequado para fronteiras de decisão axis-aligned

O ganho real do deep learning emerge em:
- Dados de alta dimensão (texto, imagem, áudio)
- Amostras massivas (milhões de exemplos)
- Tarefas de representação aprendida

Pretender vitória do MLP em todo contexto é **má prática científica**.

### Próxima sessão (09): Distribuição Diamétrica

Voltamos a um problema híbrido: ajustar a distribuição diamétrica (parametrizada via Weibull) e comparar com um MLP que aprende diretamente o histograma de DAPs. Cenário onde a forma funcional clássica é flexível mas restrita — espaço para o ML mostrar valor de novo.